In [2]:
import os
from twilio.rest import Client
from twilio_conf import TWILIO_ACCOUNT_SID, TWILIO_AUTH_TOKEN, PHONE_NUMBER, API_KEY_WAPI, APP_ACCOUNT_PAS, CORREO
import time
from requests import Request, Session
from requests.exceptions import (ConnectionError, Timeout, TooManyRedirects)
import json
import pandas as pd
import smtplib
from email.mime.text import MIMEText
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm
from datetime import datetime

In [3]:

query = "Coacalco"
api_key = API_KEY_WAPI

url_clima = f"http://api.weatherapi.com/v1/forecast.json?key={api_key}&q={query}&days=1&aqi=no&alerts=no"


In [4]:
responde = requests.get(url_clima).json()
responde

{'location': {'name': 'Coacalco',
  'region': 'México',
  'country': 'Mexico',
  'lat': 19.6345,
  'lon': -99.1005,
  'tz_id': 'America/Mexico_City',
  'localtime_epoch': 1794897983,
  'localtime': '2026-11-17 00:46'},
 'current': {'last_updated_epoch': 1794897000,
  'last_updated': '2026-11-17 00:30',
  'temp_c': 12.9,
  'temp_f': 55.2,
  'is_day': 0,
  'condition': {'text': 'Patchy rain nearby',
   'icon': '//cdn.weatherapi.com/weather/64x64/night/176.png',
   'code': 1063},
  'wind_mph': 3.6,
  'wind_kph': 5.8,
  'wind_degree': 26,
  'wind_dir': 'NNE',
  'pressure_mb': 1022.0,
  'pressure_in': 30.17,
  'precip_mm': 0.01,
  'precip_in': 0.0,
  'humidity': 79,
  'cloud': 92,
  'feelslike_c': 11.9,
  'feelslike_f': 53.4,
  'windchill_c': 12.9,
  'windchill_f': 55.1,
  'heatindex_c': 12.9,
  'heatindex_f': 55.2,
  'dewpoint_c': 9.3,
  'dewpoint_f': 48.8,
  'vis_km': 10.0,
  'vis_miles': 6.0,
  'uv': 0.0,
  'gust_mph': 7.5,
  'gust_kph': 12.1,
  'will_it_rain': 0,
  'chance_of_rain': 23,

In [5]:
responde.keys()

dict_keys(['location', 'current', 'forecast'])

In [6]:
responde['forecast']['forecastday'][0].keys()

dict_keys(['date', 'date_epoch', 'day', 'astro', 'hour'])

In [7]:
len(responde['forecast']['forecastday'][0]['hour'])

24

In [8]:
responde['forecast']['forecastday'][0]['hour'][1]['time'].split()[0] #Fecha

'2026-11-17'

In [9]:
int(responde['forecast']['forecastday'][0]['hour'][1]['time'].split()[1].split(':')[0]) #Hora

1

In [10]:
responde['forecast']['forecastday'][0]['hour'][0]['condition']['text'] #condicion

'Patchy rain nearby'

In [11]:
responde['forecast']['forecastday'][0]['hour'][0]['temp_c'] #Temperatura    

13.3

In [12]:
responde['forecast']['forecastday'][0]['hour'][0]['will_it_rain'] 

0

In [13]:
responde['forecast']['forecastday'][0]['hour'][0]['chance_of_rain']

21

# DataFrame

In [14]:
def get_forecast(responde,i):
    fecha = responde['forecast']['forecastday'][0]['hour'][i]['time'].split()[0] #Fecha
    hora = int(responde['forecast']['forecastday'][0]['hour'][i]['time'].split()[1].split(':')[0]) #Hora
    condicion = responde['forecast']['forecastday'][0]['hour'][i]['condition']['text'] #condicion
    tempe = responde['forecast']['forecastday'][0]['hour'][i]['temp_c'] #Temperatura  
    rain = responde['forecast']['forecastday'][0]['hour'][i]['will_it_rain'] 
    prob_rain = responde['forecast']['forecastday'][0]['hour'][i]['chance_of_rain']



    return fecha,hora,condicion,tempe,rain,prob_rain

In [15]:
datos = []

for i in tqdm(range(len(responde['forecast']['forecastday'][0]['hour'])), colour = 'green'):
    datos.append(get_forecast(responde,i))

100%|██████████| 24/24 [00:00<00:00, 120844.29it/s]


In [16]:
datos[10]

('2026-11-17', 10, 'Overcast', 15.5, 0, 14)

In [17]:
col = ['Fecha','Hora','Condicion','Temperatura','Lluvia','Prob_Lluvia']

df = pd.DataFrame(datos,columns=col)
df

,Fecha,Hora,Condicion,Temperatura,Lluvia,Prob_Lluvia
0,2026-11-17,0,Patchy rain nearby,13.3,0,21
1,2026-11-17,1,Patchy rain nearby,12.9,0,23
2,2026-11-17,2,Patchy rain nearby,12.5,0,24
3,2026-11-17,3,Overcast,12.2,0,25
4,2026-11-17,4,Cloudy,11.8,0,23
5,2026-11-17,5,Cloudy,11.5,0,22
6,2026-11-17,6,Overcast,11.2,0,26
7,2026-11-17,7,Overcast,12.1,0,23
8,2026-11-17,8,Overcast,13.1,0,19
9,2026-11-17,9,Overcast,14.1,0,16


In [18]:
df_rain = df[(df['Lluvia']==1) & (df['Hora']>6) & (df['Hora']< 22)]
df_rain = df_rain[['Hora','Condicion']]
df_rain = df_rain.set_index('Hora')
df_rain

,Condicion
Hora,


# MENSAJE -  TEMPLATE

In [19]:
'\nHola! \n\n\n El prononostico del tiempo hoy '+ df['Fecha'][0] + ' en ' + query +' es : \n\n\n' + str(df_rain)

'\nHola! \n\n\n El prononostico del tiempo hoy 2026-11-17 en Coacalco es : \n\n\nEmpty DataFrame\nColumns: [Condicion]\nIndex: []'

In [20]:
PHONE_NUMBER

'+525561830523'

In [23]:
mensaje = '\nHola! \n\n\n El pronostico de lluvia hoy '+ df['Fecha'][0] +' en ' + query +' es : \n\n\n' + str(df_rain)

correo = MIMEText(mensaje)
correo['Subject'] = 'Pronóstico de lluvia'
correo['From'] = CORREO
correo['To'] = 'antoniotorresj15@gmail.com'

server = smtplib.SMTP('smtp.gmail.com', 587)
server.starttls()
server.login(CORREO, APP_ACCOUNT_PAS)
server.send_message(correo)
server.quit()

print('Mensaje Enviado')

Mensaje Enviado
